# TGFusion Training — IEEE EIT 2026
**Transformer-GAN Hybrid for Multi-Modal Medical Image Fusion**  
Chaitanya Krishna Kasaraneni · Sarmista Thalapaneni

### Setup order:
1. **Runtime → Change runtime type → T4 GPU**
2. Update `GITHUB_REPO` in Cell 2
3. Run Cell 1 → Cell 2 → Cell 3 → Cell 3b (after restart) → Cell 4 onward

## Cell 1 — Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {gpu}  ({mem:.1f} GB)')
else:
    raise RuntimeError('No GPU — go to Runtime → Change runtime type → GPU')

## Cell 2 — Set paths

In [ ]:
import os

# ── UPDATE THIS ───────────────────────────────────────────────
GITHUB_REPO = 'https://github.com/chaitanyakasaraneni/tgfusion.git'
# ──────────────────────────────────────────────────────────────

REPO_DIR   = '/content/tgfusion'
AANLIB_DIR = f'{REPO_DIR}/data/aanlib'
OUTPUT_DIR = '/content/tgfusion_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Repo     → {REPO_DIR}')
print(f'Data     → {AANLIB_DIR}')
print(f'Outputs  → {OUTPUT_DIR}')

## Cell 3 — Clone repo, fix NumPy, restart runtime
> Runtime will **restart automatically** after this cell.  
> Then run **Cell 3b** to continue.

In [ ]:
import os

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -q 'numpy>=1.24.0,<2.0.0' scikit-image Pillow tqdm

print('Restarting runtime to apply numpy fix...')
os.kill(os.getpid(), 9)

## Cell 3b — Re-run after restart

In [ ]:
import os, sys

GITHUB_REPO = 'https://github.com/chaitanyakasaraneni/tgfusion.git'
REPO_DIR   = '/content/tgfusion'
AANLIB_DIR = f'{REPO_DIR}/data/aanlib'
OUTPUT_DIR = '/content/tgfusion_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}

import torch, numpy as np
assert np.__version__ < '2', f'NumPy {np.__version__} still too new — re-run Cell 3'
print(f'✓ PyTorch {torch.__version__}')
print(f'✓ NumPy   {np.__version__}')
print(f'✓ CUDA    {torch.version.cuda}')
print(f'✓ GPU     {torch.cuda.get_device_name(0)}')

from pathlib import Path
ct  = len(list(Path(f'{AANLIB_DIR}/ct_mri').glob('*/ct.png')))
pet = len(list(Path(f'{AANLIB_DIR}/mri_pet').glob('*/mri.png')))
assert ct > 0 and pet > 0, f'No data found at {AANLIB_DIR}'
print(f'✓ aanlib/ct_mri  → {ct} pairs')
print(f'✓ aanlib/mri_pet → {pet} pairs')
print('\nReady to train ✓')

## Cell 4 — Verify data splits

In [ ]:
from data.dataset import AANLIBCTMRIDataset, AANLIBMRIPETDataset

for name, cls in [('CT-MRI', AANLIBCTMRIDataset), ('MRI-PET', AANLIBMRIPETDataset)]:
    train = cls(AANLIB_DIR, split='train')
    val   = cls(AANLIB_DIR, split='val')
    test  = cls(AANLIB_DIR, split='test')
    ti = {p[0] for p in train.pairs}
    vi = {p[0] for p in val.pairs}
    si = {p[0] for p in test.pairs}
    tv = len(ti & vi); tt = len(ti & si)
    ok = '✓' if tv == 0 and tt == 0 else '✗ LEAKAGE'
    print(f'{ok}  {name}:  train={len(train.pairs)}  val={len(val.pairs)}  '
          f'test={len(test.pairs)}  overlap={tv}/{tt}')

## Cell 5 — Train CT-MRI  (Table I)
~3–4 hrs on T4  ·  ~1.5 hrs on A100

In [ ]:
%cd {REPO_DIR}
!python scripts/train_gpu.py \
    --dataset    ct_mri \
    --data_dir   {AANLIB_DIR} \
    --epochs     100 \
    --batch_size 4 \
    --output_dir {OUTPUT_DIR}

## Cell 6 — Train MRI-PET  (Table II)
Run after Cell 5 finishes

In [ ]:
%cd {REPO_DIR}
!python scripts/train_gpu.py \
    --dataset    mri_pet \
    --data_dir   {AANLIB_DIR} \
    --epochs     100 \
    --batch_size 4 \
    --output_dir {OUTPUT_DIR}

## Cell 7 — Backup checkpoints to Drive
Run after each session — `/content/` is wiped on full runtime reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DRIVE_CKPT = '/content/drive/MyDrive/tgfusion_outputs'
shutil.copytree(OUTPUT_DIR, DRIVE_CKPT, dirs_exist_ok=True)
print(f'✓ Backed up to {DRIVE_CKPT}')

## Cell 8 — Resume after disconnect
Re-run Cell 3b first, then run this cell.

In [ ]:
DATASET   = 'ct_mri'   # or 'mri_pet'
CKPT_PATH = f'{OUTPUT_DIR}/{DATASET}/checkpoints/best.pt'

# If runtime fully reset, restore from Drive first:
# from google.colab import drive; drive.mount('/content/drive')
# import shutil
# shutil.copytree('/content/drive/MyDrive/tgfusion_outputs', OUTPUT_DIR, dirs_exist_ok=True)

import torch
ckpt = torch.load(CKPT_PATH, map_location='cpu')
print(f'Resuming from epoch {ckpt["epoch"]}  best_ssim={ckpt["best_ssim"]:.4f}')

%cd {REPO_DIR}
!python scripts/train_gpu.py \
    --dataset    {DATASET} \
    --data_dir   {AANLIB_DIR} \
    --epochs     100 \
    --batch_size 4 \
    --output_dir {OUTPUT_DIR} \
    --resume     {CKPT_PATH}

## Cell 9 — Evaluate & generate LaTeX tables
Run after both tasks complete.

In [ ]:
%cd {REPO_DIR}
!python scripts/evaluate.py \
    --ckpt_ct_mri  {OUTPUT_DIR}/ct_mri/checkpoints/best.pt \
    --ckpt_mri_pet {OUTPUT_DIR}/mri_pet/checkpoints/best.pt \
    --data_dir     {AANLIB_DIR} \
    --latex

## Cell 10 — Plot training curves

In [ ]:
import re, os
import matplotlib.pyplot as plt

def parse_log(log_path):
    epochs, ssim, psnr, g_loss, d_loss = [], [], [], [], []
    with open(log_path) as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if 'Val    SSIM=' in line:
            ep_m = re.search(r'Epoch (\d+)/', lines[i-1] if i else line)
            sm   = re.search(r'SSIM=([\d.]+)', line)
            pm   = re.search(r'PSNR=([\d.]+)', line)
            if ep_m and sm and pm:
                epochs.append(int(ep_m.group(1)))
                ssim.append(float(sm.group(1)))
                psnr.append(float(pm.group(1)))
        if 'Train  G_total=' in line:
            gm = re.search(r'G_total=(-?[\d.]+)', line)
            dm = re.search(r'D_total=([\d.]+)', line)
            if gm and dm:
                g_loss.append(float(gm.group(1)))
                d_loss.append(float(dm.group(1)))
    return epochs, ssim, psnr, g_loss[:len(epochs)], d_loss[:len(epochs)]

TARGETS = {'ct_mri': (0.892, 34.7), 'mri_pet': (0.863, 32.9)}

for task in ['ct_mri', 'mri_pet']:
    log_path = f'{OUTPUT_DIR}/{task}/train.log'
    if not os.path.exists(log_path): print(f'No log for {task}'); continue
    epochs, ssim, psnr, g_loss, d_loss = parse_log(log_path)
    if not epochs: print(f'No data in {task} log'); continue

    t_ssim, t_psnr = TARGETS[task]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'TGFusion — {task.replace("_","-").upper()}', fontsize=13, fontweight='bold')

    axes[0].plot(epochs, ssim, 'b-o', ms=3, label='TGFusion')
    axes[0].axhline(t_ssim, color='r', ls='--', alpha=0.6, label=f'Paper target {t_ssim}')
    axes[0].set_title('Val SSIM'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, psnr, 'g-o', ms=3, label='TGFusion')
    axes[1].axhline(t_psnr, color='r', ls='--', alpha=0.6, label=f'Paper target {t_psnr}')
    axes[1].set_title('Val PSNR (dB)'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, g_loss, 'r-o', ms=3, label='Generator')
    axes[2].plot(epochs, d_loss, 'k-o', ms=3, label='Discriminator')
    axes[2].set_title('Train Loss'); axes[2].set_xlabel('Epoch')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{task}_curves.png', dpi=150)
    plt.show()
    print(f'Epoch {epochs[-1]}  SSIM={ssim[-1]:.4f}  PSNR={psnr[-1]:.2f} dB')